# Binance Spot hourly klines for the initial USDT universe

This notebook collects closed `1h` Binance Spot klines for the expanded 40-pair universe approved in `docs/decisions/hourly-data-collection-universe.md`. The requested period is fixed from `2017-01-01 00:00:00 UTC` through `2026-09-01 23:59:59.999 UTC`. Each pair starts at its first available Binance candle; observations are not synthesized before listing or across missing intervals.

For each pair, the notebook creates a dedicated directory under `data/raw/binance/spot/1h/`. The canonical raw artifact is a `.csv.gz` file that preserves Binance decimal fields as text. A derived Excel review copy contains `Klines`, `Metadata`, `Validation`, and `Missing Intervals` sheets, and a JSON sidecar records checksums and validation results. A timestamped collection manifest summarizes the execution. Final artifacts are validated and never overwritten.

## Research limitation

The universe was selected ex post using activity observed through 2026-09-01 and current symbol status. These data may support exploration and later protocol development, but the fixed universe must not be presented as historically known at earlier portfolio decision timestamps.

In [5]:
import importlib
from pathlib import Path
import sys

import pandas as pd


def find_repository_root(start: Path | None = None) -> Path:
    """Find the repository root from a notebook or project directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")


REPOSITORY_ROOT = find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import tcc_crypto.binance_hourly_collection as collection_module

collection_module = importlib.reload(collection_module)
load_collection_config = collection_module.load_collection_config
run_collection = collection_module.run_collection

CONFIG_PATH = (
    REPOSITORY_ROOT
    / "configs"
    / "binance_selected_usdt_hourly_collection.json"
)
config = load_collection_config(CONFIG_PATH)
NEW_SYMBOLS = (
    "HBARUSDT",
    "VETUSDT",
    "ETCUSDT",
    "ATOMUSDT",
    "SANDUSDT",
    "ENAUSDT",
    "ICPUSDT",
    "CRVUSDT",
    "MANAUSDT",
    "CHZUSDT",
)
print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Configuration: {CONFIG_PATH}")
print(f"New-pair collection scope: {', '.join(NEW_SYMBOLS)}")

Repository root: /Users/marcelapsilverio/projetos/tcc_crypto
Configuration: /Users/marcelapsilverio/projetos/tcc_crypto/configs/binance_selected_usdt_hourly_collection.json
New-pair collection scope: HBARUSDT, VETUSDT, ETCUSDT, ATOMUSDT, SANDUSDT, ENAUSDT, ICPUSDT, CRVUSDT, MANAUSDT, CHZUSDT


In [6]:
configuration_summary = pd.DataFrame(
    [
        ("Collection ID", config.collection_id),
        ("Market", config.market),
        ("Interval", config.interval),
        ("Requested start (UTC)", config.start_time_utc.isoformat()),
        ("Requested end (UTC)", config.end_time_utc.isoformat()),
        ("Selected pairs", len(config.symbols)),
        ("Output directory", str(REPOSITORY_ROOT / config.output_directory)),
        ("Source endpoint", f"{config.base_url}{config.kline_path}"),
    ],
    columns=["Field", "Value"],
)
display(configuration_summary)
display(
    pd.DataFrame(
        {
            "Selection Rank": range(1, len(config.symbols) + 1),
            "Trading Pair": config.symbols,
        }
    )
)

,Field,Value
0,Collection ID,binance_selected_usdt_hourly_2017_2026-09-01
1,Market,Binance Spot
2,Interval,1h
3,Requested start (UTC),2017-01-01T00:00:00+00:00
4,Requested end (UTC),2026-09-01T23:59:59.999000+00:00
5,Selected pairs,40
6,Output directory,/Users/marcelapsilverio/projetos/tcc_crypto/da...
7,Source endpoint,https://data-api.binance.vision/api/v3/klines


,Selection Rank,Trading Pair
0,1,BTCUSDT
1,2,FLOKIUSDT
2,3,ETHUSDT
3,4,BONKUSDT
4,5,SOLUSDT
5,6,XRPUSDT
6,7,SHIBUSDT
7,8,DOGEUSDT
8,9,BNBUSDT
9,10,ADAUSDT


## Run the collection

The following cell collects only the 10 pairs added in the approved expansion: `HBARUSDT`, `VETUSDT`, `ETCUSDT`, `ATOMUSDT`, `SANDUSDT`, `ENAUSDT`, `ICPUSDT`, `CRVUSDT`, `MANAUSDT`, and `CHZUSDT`. Collection resumes after the last page persisted in each pair checkpoint, while ranks 31 through 40 are preserved in the new manifest. Failed pairs are recorded without deleting successful outputs.

In [7]:
collection_results, manifest_path = run_collection(
    config,
    REPOSITORY_ROOT,
    symbols=NEW_SYMBOLS,
    progress_callback=print,
)
display(collection_results)
print(f"Collection manifest: {manifest_path}")

Completed 1/10: HBARUSDT (COLLECTED)
Completed 2/10: VETUSDT (COLLECTED)
Completed 3/10: ETCUSDT (COLLECTED)
Completed 4/10: ATOMUSDT (COLLECTED)
Completed 5/10: ENAUSDT (COLLECTED)
Completed 6/10: SANDUSDT (COLLECTED)
Completed 7/10: ICPUSDT (COLLECTED)
Completed 8/10: CRVUSDT (COLLECTED)
Completed 9/10: MANAUSDT (COLLECTED)
Completed 10/10: CHZUSDT (COLLECTED)


,Selection Rank,Trading Pair,Status,Rows,Missing Hours,Shortened Candles,First Open (UTC),Last Close (UTC),Raw CSV,Raw CSV SHA-256,Workbook,Workbook SHA-256,Metadata,Error
0,31,HBARUSDT,COLLECTED,60680,36,8,2019-09-29T04:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,a46deabbcbbb39bd47bf4a9d263570251f425664474e61...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,9d38dcb65fad3a0e3f9558f3b932d11fa37419919fe2c2...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
1,32,VETUSDT,COLLECTED,70990,70,8,2018-07-25T04:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,c18cf865c47d7646b399bc45f4db3102a6736faddc47bf...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,f3214e5eec186ad1c5e2cd90e13b3a0032d1fc40a3e0c8...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
2,33,ETCUSDT,COLLECTED,72006,88,10,2018-06-12T02:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,385764a0b3f8008d91acca5cb8e3f178de6d7bfd27e36d...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,a82566f9af560bd11a5e6046e408cc68f394fcc3769ad7...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
3,34,ATOMUSDT,COLLECTED,64334,54,9,2019-04-29T04:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,f5b33d0e97647aafd3cb899f18b59154369cabb29bf736...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,670eb41769523be746ae813da41b809648f617b45596b5...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
4,35,SANDUSDT,COLLECTED,53007,20,5,2020-08-14T13:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,f9974aaab846456124fea52cd9cf7d5cba0967b8319c1d...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,5f146ee42f98252943ec6fc3ebf4739ea049d25e4108ca...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
5,36,ENAUSDT,COLLECTED,21184,0,0,2024-04-02T08:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,f870ab80a8cf41f6f703fd091d85f7fbe1f1d32470f0e9...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,f255bb051943fe4cd773e3dde2fcd6ec438c9d261d7323...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
6,37,ICPUSDT,COLLECTED,46552,7,3,2021-05-11T01:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,7e2d6b9ba0a6561d7b2951581bbf256a24365140e2917c...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,4dca161b208ce9877aa536cc98ec5df1fe2166333bf3b7...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
7,38,CRVUSDT,COLLECTED,52992,20,6,2020-08-15T04:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,505184b65493ee01bdd697bfc2c92df76158fa65677d16...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,8f90ee84a98494e28b7dfa8062b9eab8a89304832b077a...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
8,39,MANAUSDT,COLLECTED,53202,20,6,2020-08-06T10:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,348a072efc5b025d94f7870e84513cc0778765b845f11f...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,85fc5331677749ba6fbee718fe04d0db0c0394f8c5a31e...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None
9,40,CHZUSDT,COLLECTED,61232,36,8,2019-09-06T04:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,0dc71058b0efa3ce5812de7431df23052bbf6db2bc839b...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,1e17c8e38be92821ea8f0f90aa17bbe63d979ac928bb7a...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,None


Collection manifest: /Users/marcelapsilverio/projetos/tcc_crypto/data/raw/binance/spot/1h/collection_manifest_20260905T210702563723Z.xlsx


In [8]:
status_summary = (
    collection_results["Status"]
    .value_counts(dropna=False)
    .rename_axis("Status")
    .reset_index(name="Pairs")
)
display(status_summary)

failed = collection_results.loc[collection_results["Status"] == "FAILED"]
pairs_requiring_gap_review = collection_results.loc[
    collection_results["Missing Hours"].fillna(0) > 0
]
if not pairs_requiring_gap_review.empty:
    print("Pairs with missing hourly intervals requiring review:")
    display(
        pairs_requiring_gap_review[
            ["Trading Pair", "Missing Hours", "Workbook"]
        ]
    )

pairs_with_shortened_candles = collection_results.loc[
    collection_results["Shortened Candles"].fillna(0) > 0
]
if not pairs_with_shortened_candles.empty:
    print("Pairs with shortened historical candles requiring review:")
    display(
        pairs_with_shortened_candles[
            ["Trading Pair", "Shortened Candles", "Workbook"]
        ]
    )

if not failed.empty:
    display(failed[["Trading Pair", "Error"]])
    raise RuntimeError(
        f"Collection finished with {len(failed)} failed pair(s). "
        "Review the manifest and rerun the collection cell."
    )

print("All requested pairs were collected or already existed.")

,Status,Pairs
0,COLLECTED,10


Pairs with missing hourly intervals requiring review:


,Trading Pair,Missing Hours,Workbook
0,HBARUSDT,36,/Users/marcelapsilverio/projetos/tcc_crypto/da...
1,VETUSDT,70,/Users/marcelapsilverio/projetos/tcc_crypto/da...
2,ETCUSDT,88,/Users/marcelapsilverio/projetos/tcc_crypto/da...
3,ATOMUSDT,54,/Users/marcelapsilverio/projetos/tcc_crypto/da...
4,SANDUSDT,20,/Users/marcelapsilverio/projetos/tcc_crypto/da...
6,ICPUSDT,7,/Users/marcelapsilverio/projetos/tcc_crypto/da...
7,CRVUSDT,20,/Users/marcelapsilverio/projetos/tcc_crypto/da...
8,MANAUSDT,20,/Users/marcelapsilverio/projetos/tcc_crypto/da...
9,CHZUSDT,36,/Users/marcelapsilverio/projetos/tcc_crypto/da...


Pairs with shortened historical candles requiring review:


,Trading Pair,Shortened Candles,Workbook
0,HBARUSDT,8,/Users/marcelapsilverio/projetos/tcc_crypto/da...
1,VETUSDT,8,/Users/marcelapsilverio/projetos/tcc_crypto/da...
2,ETCUSDT,10,/Users/marcelapsilverio/projetos/tcc_crypto/da...
3,ATOMUSDT,9,/Users/marcelapsilverio/projetos/tcc_crypto/da...
4,SANDUSDT,5,/Users/marcelapsilverio/projetos/tcc_crypto/da...
6,ICPUSDT,3,/Users/marcelapsilverio/projetos/tcc_crypto/da...
7,CRVUSDT,6,/Users/marcelapsilverio/projetos/tcc_crypto/da...
8,MANAUSDT,6,/Users/marcelapsilverio/projetos/tcc_crypto/da...
9,CHZUSDT,8,/Users/marcelapsilverio/projetos/tcc_crypto/da...


All requested pairs were collected or already existed.


## Interpretation and next checks

A missing interval inside the observed span is recorded as `REVIEW`; it is not filled in the canonical raw CSV. A candle whose close remains inside its nominal hour but occurs before the final millisecond is also preserved and marked `REVIEW`, with details in `Nonstandard Durations`. A close before the open or beyond the nominal hour remains a fatal validation failure. Time before a pair's first observed candle is treated as pre-listing history rather than missing data. The Excel workbooks are derived inspection artifacts and must not replace the CSV files as pipeline inputs.